# META-CXR Encoder Mean F1 Table

Notebook này chỉ tính bảng **Mean F1 Score across 5 common abnormalities** cho các cấu hình encoder, dùng checkpoint từ Kaggle dataset `mimic-cxr-checkpoint`.

Kaggle datasets cần attach:
- `mimic-cxr-checkpoint` chứa checkpoint các run.
- `mimic-cxr-jpg-lite` chứa ảnh + CheXpert CSV.
- `mimic-cxr-p10-processed` chứa `train.csv`, `val.csv`, `test.csv`.
- Dataset/source chứa code `META-CXR` nếu notebook không nằm sẵn trong repo.

Kết quả cuối cùng là một bảng 4 cột: `RN50`, `ViT`, `Swin`, `Mean F1 Score`.

In [ ]:
# Kaggle: bật Internet nếu môi trường chưa có đủ dependency/cache model.
# Không pin transformers==4.30.2 ở Kaggle mới vì nó kéo tokenizers<0.14 và có thể phải build từ source.
# Code Qformer hiện đã fallback sang transformers.pytorch_utils nên dùng transformers sẵn có của Kaggle là ổn.
!pip install -q \
  "numpy<2" \
  "opencv-python<4.10" \
  "omegaconf==2.3.0" \
  iopath timm pandas scikit-image accelerate sentencepiece protobuf \
  iterative-stratification einops fairscale pycocoevalcap webdataset decord \
  ftfy regex hi-ml-multimodal torchinfo

In [ ]:
import os
import sys
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')

def is_meta_cxr_project(path: Path) -> bool:
    return (path / 'model' / 'lavis').exists() and (path / 'pretraining').exists()

project_candidates = [Path.cwd(), WORK_DIR / 'META-CXR']
if INPUT_DIR.exists():
    for root in INPUT_DIR.glob('*'):
        project_candidates.extend([root, root / 'META-CXR'])

source_project = next((p for p in project_candidates if is_meta_cxr_project(p)), None)
if source_project is None:
    raise FileNotFoundError('Không tìm thấy code META-CXR. Hãy attach dataset/source chứa thư mục META-CXR.')

PROJECT_DIR = WORK_DIR / 'META-CXR'
if source_project.resolve() != PROJECT_DIR.resolve():
    if PROJECT_DIR.exists() and is_meta_cxr_project(PROJECT_DIR):
        pass
    else:
        ignore = shutil.ignore_patterns('.git', 'wandb', '__pycache__', '*.pyc', 'output', 'outputs', 'checkpoints')
        shutil.copytree(source_project, PROJECT_DIR, dirs_exist_ok=True, ignore=ignore)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'model'))

def first_existing(paths):
    for item in paths:
        path = Path(item)
        if path.exists():
            return path
    raise FileNotFoundError('Không tìm thấy path nào trong: ' + ', '.join(map(str, paths)))

IMAGE_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-jpg-lite',
    '/kaggle/input/datasets/mimic-cxr-jpg-lite',
])
PROCESSED_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-p10-processed',
    '/kaggle/input/datasets/mimic-cxr-p10-processed',
])
CHECKPOINT_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-checkpoint',
    '/kaggle/input/meta-cxr-checkpoint',
    '/kaggle/input/meta-cxr-checkpoints',
])

(PROJECT_DIR / 'configs').mkdir(exist_ok=True)
(PROJECT_DIR / 'configs' / 'env_config.yaml').write_text(f'''paths:
  data_root: "{IMAGE_ROOT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"
  chexpert_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "/kaggle/working/output"
  checkpoint_dir: "{CHECKPOINT_ROOT}"
wandb:
  entity: ""
  project: "meta-cxr-encoder-comparison"
java:
  home: "/usr/lib/jvm/java-11-openjdk-amd64"
  path: "/usr/lib/jvm/java-11-openjdk-amd64/bin:"
''')

print('PROJECT_DIR    =', PROJECT_DIR)
print('IMAGE_ROOT     =', IMAGE_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)

In [ ]:
import gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

# Registration imports.
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler
from model.lavis.datasets.builders import *
from model.lavis.models import *
from model.lavis.processors import *
from model.lavis.tasks import *
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from local_config import VIS_ROOT

registry.mapping['paths']['cache_root'] = '.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_BATCH_SIZE = 4
NUM_WORKERS = 2

CHEXPERT_COLS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
]

FIVE_COMMON_ABNORMALITIES = [
    'Atelectasis',
    'Cardiomegaly',
    'Consolidation',
    'Edema',
    'Pleural Effusion',
]
TASK_IDXS = [CHEXPERT_COLS.index(name) for name in FIVE_COMMON_ABNORMALITIES]

print('DEVICE =', DEVICE)
print('5 tasks =', FIVE_COMMON_ABNORMALITIES)

In [ ]:
TABLE_RUNS = [
    {'run': '01_biovil_only',        'RN50': True,  'ViT': False, 'Swin': False},
    {'run': '02_pubmedclip_only',    'RN50': False, 'ViT': True,  'Swin': False},
    {'run': '03_swin_only',          'RN50': False, 'ViT': False, 'Swin': True},
    {'run': '04_biovil_pubmedclip',  'RN50': True,  'ViT': True,  'Swin': False},
    {'run': '05_biovil_swin',        'RN50': True,  'ViT': False, 'Swin': True},
    {'run': '07_all_three',          'RN50': True,  'ViT': True,  'Swin': True},
]

def find_checkpoint(run_name: str) -> Path:
    best = [p for p in CHECKPOINT_ROOT.rglob('checkpoint_best.pth') if run_name in str(p)]
    if best:
        return sorted(best, key=lambda p: len(str(p)))[0]
    last = [p for p in CHECKPOINT_ROOT.rglob('checkpoint_last.pth') if run_name in str(p)]
    if last:
        print(f'WARNING: {run_name}: checkpoint_best.pth not found, using checkpoint_last.pth')
        return sorted(last, key=lambda p: len(str(p)))[0]
    raise FileNotFoundError(f'Không tìm thấy checkpoint_best/last cho run {run_name} trong {CHECKPOINT_ROOT}')

def build_cfg(run_name: str):
    cfg_path = PROJECT_DIR / 'pretraining' / 'configs' / 'encoder_comparison' / f'{run_name}.yaml'
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)

def build_model_for_run(run_name: str, checkpoint_path: Path):
    cfg = build_cfg(run_name)
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'{run_name}: loaded {checkpoint_path.name}; missing={len(missing)}, unexpected={len(unexpected)}')
    model.to(DEVICE)
    model.eval()
    return cfg, model

def make_test_loader(cfg):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split='test',
        cfg=cfg,
        truncate=None,
    )
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
    )

@torch.no_grad()
def predict_logits_with_text(model, batch):
    image = batch['image'].to(DEVICE, non_blocking=True)
    text = batch['text_output']

    cnn_patches, vit_patches, swin_patches, _ = model._encode_image_streams(image, apply_aug=False)
    text_tokens = model.tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=model.max_txt_len,
        return_tensors='pt',
    ).to(DEVICE)
    text_output = model.Qformer.bert(
        text_tokens.input_ids,
        attention_mask=text_tokens.attention_mask,
        return_dict=True,
    )
    logits, _, _, _, _ = model.mhcac(
        cnn_patches=cnn_patches,
        vit_patches=vit_patches,
        swin_patches=swin_patches,
        text_embeddings=text_output.last_hidden_state,
        labels=None,
    )
    return logits

def mean_f1_for_run(run_name: str):
    checkpoint_path = find_checkpoint(run_name)
    cfg, model = build_model_for_run(run_name, checkpoint_path)
    loader = make_test_loader(cfg)

    all_preds = []
    all_labels = []
    for batch in tqdm(loader, desc=run_name):
        logits = predict_logits_with_text(model, batch)
        preds = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)
        all_preds.append(preds[:, TASK_IDXS].cpu().numpy())
        all_labels.append(batch['classification_labels'][:, TASK_IDXS].cpu().numpy())

    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_labels, axis=0)

    per_task = {}
    for col_idx, task_name in enumerate(FIVE_COMMON_ABNORMALITIES):
        per_task[task_name] = f1_score(
            y_true[:, col_idx],
            y_pred[:, col_idx],
            average='weighted',
            zero_division=1,
        )
    mean_f1 = float(np.mean(list(per_task.values())))

    del model, loader
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return mean_f1, per_task, checkpoint_path

In [ ]:
rows = []
details = {}

for item in TABLE_RUNS:
    mean_f1, per_task, checkpoint_path = mean_f1_for_run(item['run'])
    details[item['run']] = {'mean_f1': mean_f1, 'per_task': per_task, 'checkpoint': str(checkpoint_path)}
    rows.append({
        'RN50': '✓' if item['RN50'] else '–',
        'ViT': '✓' if item['ViT'] else '–',
        'Swin': '✓' if item['Swin'] else '–',
        'Mean F1 Score': mean_f1,
    })

table = pd.DataFrame(rows, columns=['RN50', 'ViT', 'Swin', 'Mean F1 Score'])
table.to_csv('/kaggle/working/encoder_mean_f1_table.csv', index=False)

display(
    table.style
    .hide(axis='index')
    .format({'Mean F1 Score': '{:.3f}'})
    .set_caption('Mean F1 Score Across 5 Common Abnormalities on the MIMIC-CXR Test Set')
)

table